## Organize reachable leisure pois

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
# Load libs
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm
import time
from lib import helpers as helpers
import numpy as np

## 1. Load w-k results and process
Reachable POIs, and the travel times from workplace to them.

- Situation 1: Real time budget
- Situation 2: 90 min uniform time budget
- Situation 3: 90 min one-way uniform time budget

In [ ]:
df_com = pd.read_csv("dbs/data_p/commuter_time_budget.csv")
df_com.head()

In [ ]:
def process_mode(mode):
    df_list = []
    candidate_individuals = dict()
    for situation in ['01', '02', '03']:
        candidate_individuals[situation] = []
        for tbin in ['15', '30', '45', '60', '75']:
            # Load the results
            print(f'Loading mode {mode}, situation {situation}, time bin {tbin}.')
            df = pd.read_csv(f'dbs/sp_accessibility/tt_wk_{mode}_17_{situation}_{tbin}.csv')
            df = pd.merge(df, df_com[['ID', 'tt_wkh_1', 'tt_wkh_2', 'tt_wkh_3']], left_on='from_id', right_on='ID', how='left')
            for v, nv in zip(['tt_wkh_1', 'tt_wkh_2', 'tt_wkh_3'], ['tt_kh_1', 'tt_kh_2', 'tt_kh_3']):
                df[nv] = df[v] - df['travel_time_p50']
            # Collect candidate POIs and individuals
            sit = int(situation)
            candidate_individuals[situation].extend(df[df[f'tt_kh_{sit}'] > 0]['from_id'].unique().tolist())
            candidate_individuals[situation] = list(set(candidate_individuals[situation]))
            temp = df[df[f'tt_kh_{sit}'] > 0][['ID', 'to_id', f'tt_kh_{sit}']].copy().rename(columns={f'tt_kh_{sit}': 'tt_kh'})
            temp['mode'] = mode
            temp['situation'] = situation
            df_list.append(temp)
    df_p = pd.concat(df_list, ignore_index=True)
    return df_p, candidate_individuals

In [7]:
# Revised situation = 2 (90 min budget)
def process_mode_r(mode):
    df_list = []
    candidate_individuals = []
    for tbin in ['15', '30', '45', '60', '75', '90']:
        # Load the results
        print(f'Loading mode {mode}, time bin {tbin}.')
        df = pd.read_csv(f'dbs/sp_accessibility_r/tt_wk_{mode}_17_{tbin}.csv')
        df = pd.merge(df, df_com[['ID', 'tt_wkh_2']], left_on='from_id', right_on='ID', how='left')
        for v, nv in zip(['tt_wkh_2'], ['tt_kh_2']):
            df[nv] = df[v] - df['travel_time_p50']
        # Collect candidate POIs and individuals
        candidate_individuals.extend(df[df[f'tt_kh_2'] > 0]['from_id'].unique().tolist())
        candidate_individuals = list(set(candidate_individuals))
        temp = df[df[f'tt_kh_2'] > 0][['ID', 'to_id', f'tt_kh_2']].copy().rename(columns={f'tt_kh_2': 'tt_kh'})
        temp['mode'] = mode
        df_list.append(temp)
    df_p = pd.concat(df_list, ignore_index=True)
    return df_p, candidate_individuals

## 2. Work-to-k results

In [ ]:
df_p_1, candidate_individuals_1 = process_mode('pt')
for situation in ['01', '02', '03']:
    no_pois = df_p_1[df_p_1['situation'] == situation]['to_id'].nunique()
    print(f'Situation {situation}: {len(candidate_individuals_1[situation])} individuals, {no_pois} POIs.')

In [ ]:
df_p_2, candidate_individuals_2 = process_mode('car')
for situation in ['01', '02', '03']:
    no_pois = df_p_2[df_p_2['situation'] == situation]['to_id'].nunique()
    print(f'Situation {situation}: {len(candidate_individuals_2[situation])} individuals, {no_pois} POIs.')

### 2.1 Updated SPA (90 min situation 2)

In [ ]:
df_p_1, candidate_individuals_1 = process_mode_r('pt')
no_pois = df_p_1['to_id'].nunique()
print(f'Situation 2: {len(candidate_individuals_1)} individuals, {no_pois} POIs.')

In [ ]:
df_p_2, candidate_individuals_2 = process_mode_r('car')
no_pois = df_p_2['to_id'].nunique()
print(f'Situation 2: {len(candidate_individuals_1)} individuals, {no_pois} POIs.')

## 3. Data preparation for k-to-Home
### 3.1 Prepare destinations

In [10]:
df_work = pd.read_csv('dbs/data_p/commuter_trips.csv')
df_work = df_work[df_work['purpose_d'] == 'HOME'].copy()
df_work.drop_duplicates(subset=['ID'], inplace=True)
df_work = df_work[['ID', 'end_lon', 'end_lat']].rename(columns={'ID': 'id', 'end_lon': 'lon', 'end_lat': 'lat'})

In [8]:
mode = 'pt'
for situation in ['01', '02', '03']:
    df_work[df_work['id'].isin(candidate_individuals_1[situation])].to_csv(f'dbs/sp_accessibility/data/destinations_{mode}_{int(situation)}.csv', index=False)

In [9]:
mode = 'car'
for situation in ['01', '02', '03']:
    df_work[df_work['id'].isin(candidate_individuals_2[situation])].to_csv(f'dbs/sp_accessibility/data/destinations_{mode}_{int(situation)}.csv', index=False)

#### Updated

In [11]:
mode = 'pt'
df_work[df_work['id'].isin(candidate_individuals_1)].to_csv(f'dbs/sp_accessibility_r/data/destinations_{mode}.csv', index=False)
mode = 'car'
df_work[df_work['id'].isin(candidate_individuals_2)].to_csv(f'dbs/sp_accessibility_r/data/destinations_{mode}.csv', index=False)

### 3.2 Prepare origins

In [ ]:
df2save = pd.read_csv("dbs/sp_accessibility_r/data/destinations.csv")
df2save.head()

In [ ]:
df_p_2.head()

In [ ]:
# Define bins and labels (right-exclusive, so [0,15), [15,30), …)
bins   = [0, 15, 30, 45, 60, 75, 90.1]
labels = [15, 30, 45, 60, 75, 90]   # dictionary keys
mode = 'car'    # car, pt and change df_p_2 to df_p_1 for public transport
for situation in ['01', '02', '03']:
    temp = df_p_2[df_p_2['situation'] == situation].copy()
    temp['tt_kh'] = temp['tt_kh'].apply(lambda x: x if x <= 90 else 90)  # Cap at 90 minutes
    temp['group'] = pd.cut(temp['tt_kh'], bins=bins, labels=labels, right=False)
    # Build dictionary: {15: [...IDs...], 30: [...], …}
    group_dict = (
        temp.groupby('group')['to_id']
            .apply(lambda x: list(set(x)))
            .to_dict()
    )
    for lb in labels:
        df2save_lb = df2save[df2save['id'].isin(group_dict[lb])].copy()
        print("No. of POIs", len(df2save_lb))
        if len(df2save_lb) > 0:
            if (len(df2save_lb) > 10000) & (mode == 'car'):
                # Split into 8 batches
                batches = np.array_split(df2save_lb, 8)
                for i, batch in enumerate(batches, start=1):
                    out_path = f'dbs/sp_accessibility/data/origins_{mode}_{int(situation)}_{lb}_part{i}.csv'
                    batch[['id', 'lon', 'lat']].to_csv(out_path, index=False)
                    print(f"Saved batch {i} with {len(batch)} rows → {out_path}")
            else:
                # Save normally
                out_path = f'dbs/sp_accessibility/data/origins_{mode}_{int(situation)}_{lb}.csv'
                df2save_lb[['id', 'lon', 'lat']].to_csv(out_path, index=False)
                print(f"Saved single file with {len(df2save_lb)} rows → {out_path}")

### 3.3 Updated SPA (90 min situation 2)

In [ ]:
# Updated situation 2 (90 min travel time budget)
# Define bins and labels (right-exclusive, so [0,15), [15,30), …)
bins   = [0, 15, 30, 45, 60, 75, 90.1]
labels = [15, 30, 45, 60, 75, 90]   # dictionary keys
mode = 'pt'    # car, pt and change df_p_2 to df_p_1 for public transport
temp = df_p_1.copy()
temp['tt_kh'] = temp['tt_kh'].apply(lambda x: x if x <= 90 else 90)  # Cap at 90 minutes
temp['group'] = pd.cut(temp['tt_kh'], bins=bins, labels=labels, right=False)
# Build dictionary: {15: [...IDs...], 30: [...], …}
group_dict = (
    temp.groupby('group')['to_id']
        .apply(lambda x: list(set(x)))
        .to_dict()
)
for lb in labels:
    df2save_lb = df2save[df2save['id'].isin(group_dict[lb])].copy()
    print("No. of POIs", len(df2save_lb))
    if len(df2save_lb) > 0:
        if (len(df2save_lb) > 10000) & (mode == 'car'):
            # Split into 8 batches
            batches = np.array_split(df2save_lb, 8)
            for i, batch in enumerate(batches, start=1):
                out_path = f'dbs/sp_accessibility_r/data/origins_{mode}_{lb}_part{i}.csv'
                batch[['id', 'lon', 'lat']].to_csv(out_path, index=False)
                print(f"Saved batch {i} with {len(batch)} rows → {out_path}")
        else:
            # Save normally
            out_path = f'dbs/sp_accessibility_r/data/origins_{mode}_{lb}.csv'
            df2save_lb[['id', 'lon', 'lat']].to_csv(out_path, index=False)
            print(f"Saved single file with {len(df2save_lb)} rows → {out_path}")